In [2]:
!pip install yellowbrick
import warnings
warnings.filterwarnings("ignore")

import numpy as np

# ── Compatibilidade: yellowbrick 1.5 usa np.percentile(interpolation=...)
# ── que foi renomeado para `method` no numpy >= 1.22
_np_percentile_orig = np.percentile
def _np_percentile_compat(*args, interpolation=None, **kwargs):
    if interpolation is not None:
        kwargs["method"] = interpolation
    return _np_percentile_orig(*args, **kwargs)
np.percentile = _np_percentile_compat

# ── Compatibilidade: sklearn >= 1.6 faz _check_targets retornar 4 valores;
# ── o yellowbrick ClassPredictionError espera apenas 3.
# ── Patchamos apenas no namespace do yellowbrick, não no sklearn global.
from sklearn.metrics import _classification as _sk_clf_mod
_orig_check_targets = _sk_clf_mod._check_targets
def _check_targets_yb(y_true, y_pred, *args, **kwargs):
    result = _orig_check_targets(y_true, y_pred, *args, **kwargs)
    return result[:3]   # yellowbrick só usa os 3 primeiros

import importlib as _importlib
_yb_cpe = _importlib.import_module("yellowbrick.classifier.class_prediction_error")
vars(_yb_cpe)["_check_targets"] = _check_targets_yb

# ── Helper: sklearn >= 1.1 removeu _estimator_type como atributo de instância;
# ── o yellowbrick ainda precisa dele. Esta função garante compatibilidade.
def tag(estimator, etype):
    """Adiciona _estimator_type ao estimador para compatibilidade com yellowbrick."""
    estimator._estimator_type = etype
    return estimator

def clf(estimator): return tag(estimator, "classifier")
def reg(estimator): return tag(estimator, "regressor")
def clu(estimator): return tag(estimator, "clusterer")

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # renderização sem display

from sklearn.datasets import make_blobs, make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

# ─── Yellowbrick Imports ──────────────────────────────────────────────────────
# Clusterização
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer, InterclusterDistance
# Classificação
from yellowbrick.classifier import ConfusionMatrix, ROCAUC, ClassificationReport, ClassPredictionError
# Regressão
from yellowbrick.regressor import PredictionError, ResidualsPlot
# Features
from yellowbrick.features import Rank1D, Rank2D, RadViz, ParallelCoordinates, FeatureImportances

OUTPUT_DIR = "."  # altere para salvar em outro diretório


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# SEÇÃO 1 — VISUALIZADORES DE CLUSTERIZAÇÃO
# ══════════════════════════════════════════════════════════════════════════════

def secao_clusterizacao():
    """
    Bases artificiais com 3 clusters bem definidos (make_blobs).
    Demonstra KElbowVisualizer, SilhouetteVisualizer e InterclusterDistance.
    """
    print("\n" + "="*60)
    print("  SEÇÃO 1 — CLUSTERIZAÇÃO")
    print("="*60)

    # ── Dados ────────────────────────────────────────────────────────────────
    X, y_true = make_blobs(
        n_samples=300,
        centers=3,
        cluster_std=0.8,
        random_state=42
    )
    X = StandardScaler().fit_transform(X)

    # ── 1.1  KElbowVisualizer ─────────────────────────────────────────────
    print("\n[1.1] KElbowVisualizer — Método do Cotovelo")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("KElbowVisualizer — três métricas", fontsize=14, fontweight="bold")

    metricas = ["distortion", "silhouette", "calinski_harabasz"]
    titulos  = ["Distorção (Inércia)", "Coeficiente de Silhueta", "Calinski-Harabasz"]

    for ax, metrica, titulo in zip(axes, metricas, titulos):
        m = KMeans(random_state=42, n_init="auto")
        m._estimator_type = "clusterer"   # compatibilidade yellowbrick ↔ sklearn>=1.2
        viz = KElbowVisualizer(
            m,
            k=(2, 10),
            metric=metrica,
            timings=False,
            ax=ax
        )
        viz.fit(X)
        viz.finalize()
        ax.set_title(titulo, fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()
    
    # ── 1.2  SilhouetteVisualizer ─────────────────────────────────────────
    print("\n[1.2] SilhouetteVisualizer — perfil de silhueta por cluster")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("SilhouetteVisualizer — K = 2, 3, 4", fontsize=14, fontweight="bold")

    for ax, k in zip(axes, [2, 3, 4]):
        model = KMeans(n_clusters=k, random_state=42, n_init="auto")
        model._estimator_type = "clusterer"
        viz = SilhouetteVisualizer(model, colors="yellowbrick", ax=ax)
        viz.fit(X)
        viz.finalize()
        ax.set_title(f"K = {k}", fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()

    # ── 1.3  InterclusterDistance ─────────────────────────────────────────
    print("\n[1.3] InterclusterDistance — distância geométrica entre clusters")
    fig, ax = plt.subplots(figsize=(8, 7))

    model = KMeans(n_clusters=3, random_state=42, n_init="auto")
    model._estimator_type = "clusterer"
    viz = InterclusterDistance(model, ax=ax)
    viz.fit(X)
    viz.finalize()
    ax.set_title("InterclusterDistance (MDS 2D)", fontsize=12, fontweight="bold")

    plt.tight_layout()
    plt.show()

secao_clusterizacao()


  SEÇÃO 1 — CLUSTERIZAÇÃO

[1.1] KElbowVisualizer — Método do Cotovelo

[1.2] SilhouetteVisualizer — perfil de silhueta por cluster

[1.3] InterclusterDistance — distância geométrica entre clusters


RecursionError: maximum recursion depth exceeded

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEÇÃO 2 — VISUALIZADORES DE CLASSIFICAÇÃO
# ══════════════════════════════════════════════════════════════════════════════

def secao_classificacao():
    """
    Classificação multiclasse (4 classes) com RandomForestClassifier.
    Demonstra ConfusionMatrix, ROCAUC, ClassificationReport e ClassPredictionError.
    """
    print("\n" + "="*60)
    print("  SEÇÃO 2 — CLASSIFICAÇÃO")
    print("="*60)

    # ── Dados ────────────────────────────────────────────────────────────────
    X, y = make_classification(
        n_samples=600,
        n_features=12,
        n_informative=8,
        n_redundant=2,
        n_classes=4,
        n_clusters_per_class=1,
        random_state=42
    )
    classes = ["Classe A", "Classe B", "Classe C", "Classe D"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    model = clf(RandomForestClassifier(n_estimators=120, random_state=42))

    # ── 2.1  ConfusionMatrix ──────────────────────────────────────────────
    print("\n[2.1] ConfusionMatrix")
    fig, ax = plt.subplots(figsize=(7, 6))

    viz = ConfusionMatrix(model, classes=classes, ax=ax)
    viz.fit(X_train, y_train)
    viz.score(X_test, y_test)
    viz.finalize()
    ax.set_title("Matriz de Confusão", fontsize=12, fontweight="bold")

    plt.show()

     
    ax.set_title("Classification Report (Heatmap)", fontsize=12, fontweight="bold")
    plt.show()

    # ── 2.4  ClassPredictionError ─────────────────────────────────────────
    print("\n[2.4] ClassPredictionError — erros sistemáticos por classe")
    fig, ax = plt.subplots(figsize=(8, 5))

    model4 = clf(RandomForestClassifier(n_estimators=120, random_state=42))
    viz = ClassPredictionError(model4, classes=classes, ax=ax)
    viz.fit(X_train, y_train)
    viz.score(X_test, y_test)
    viz.finalize()
    ax.set_title("Class Prediction Error", fontsize=12, fontweight="bold")
    plt.show()
    
secao_classificacao()


  SEÇÃO 2 — CLASSIFICAÇÃO

[2.1] ConfusionMatrix

[2.2] ROCAUC — curvas ROC multiclasse

[2.3] ClassificationReport — mapa de calor de métricas

[2.4] ClassPredictionError — erros sistemáticos por classe


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SEÇÃO 3 — VISUALIZADORES DE REGRESSÃO
# ══════════════════════════════════════════════════════════════════════════════

def secao_regressao():
    """
    Regressão contínua com Ridge e RandomForestRegressor.
    Demonstra PredictionError e ResidualsPlot.
    """
    print("\n" + "="*60)
    print("  SEÇÃO 3 — REGRESSÃO")
    print("="*60)

    # ── Dados ────────────────────────────────────────────────────────────────
    X, y = make_regression(
        n_samples=500,
        n_features=10,
        n_informative=7,
        noise=25,
        random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # ── 3.1  PredictionError ─────────────────────────────────────────────
    print("\n[3.1] PredictionError — valores reais × preditos")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("PredictionError — Ridge vs Random Forest", fontsize=13, fontweight="bold")

    for ax, model, nome in zip(
        axes,
        [reg(Ridge()), reg(RandomForestRegressor(n_estimators=100, random_state=42))],
        ["Ridge", "Random Forest"]
    ):
        viz = PredictionError(model, ax=ax)
        viz.fit(X_train, y_train)
        viz.score(X_test, y_test)
        viz.finalize()
        ax.set_title(f"PredictionError — {nome}", fontsize=11, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/08_prediction_error.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 08_prediction_error.png")

    # ── 3.2  ResidualsPlot ────────────────────────────────────────────────
    print("\n[3.2] ResidualsPlot — homocedasticidade e overfitting")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("ResidualsPlot — treino vs teste", fontsize=13, fontweight="bold")

    for ax, model, nome in zip(
        axes,
        [reg(Ridge()), reg(RandomForestRegressor(n_estimators=100, random_state=42))],
        ["Ridge", "Random Forest"]
    ):
        viz = ResidualsPlot(model, train_color="steelblue", test_color="firebrick", ax=ax)
        viz.fit(X_train, y_train)
        viz.score(X_test, y_test)
        viz.finalize()
        ax.set_title(f"Resíduos — {nome}", fontsize=11, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/09_residuals_plot.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 09_residuals_plot.png")


# ══════════════════════════════════════════════════════════════════════════════
# SEÇÃO 4 — ANÁLISE DE ATRIBUTOS (FEATURE ANALYSIS)
# ══════════════════════════════════════════════════════════════════════════════

def secao_features():
    """
    Dados multiclasse (3 classes, 10 features) para análise de atributos.
    Demonstra Rank1D, Rank2D, RadViz, ParallelCoordinates e FeatureImportances.
    """
    print("\n" + "="*60)
    print("  SEÇÃO 4 — ANÁLISE DE ATRIBUTOS")
    print("="*60)

    # ── Dados ────────────────────────────────────────────────────────────────
    X, y = make_classification(
        n_samples=400,
        n_features=10,
        n_informative=6,
        n_redundant=2,
        n_classes=3,
        n_clusters_per_class=1,
        random_state=42
    )
    feature_names = [f"feat_{i:02d}" for i in range(10)]
    class_names   = ["Alfa", "Beta", "Gama"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    # ── 4.1  Rank1D ───────────────────────────────────────────────────────
    print("\n[4.1] Rank1D — ranking univariado das features (Shapiro-Wilk)")
    fig, ax = plt.subplots(figsize=(10, 5))

    viz = Rank1D(features=feature_names, algorithm="shapiro", ax=ax)
    viz.fit(X, y)
    viz.transform(X)
    viz.finalize()
    ax.set_title("Rank1D — Shapiro-Wilk", fontsize=11, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/10_rank1d.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 10_rank1d.png")

    # ── 4.2  Rank2D ───────────────────────────────────────────────────────
    print("\n[4.2] Rank2D — correlações bivariadas entre features")
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Rank2D", fontsize=13, fontweight="bold")

    for ax, algo in zip(axes, ["pearson", "covariance"]):
        viz = Rank2D(features=feature_names, algorithm=algo, ax=ax)
        viz.fit(X, y)
        viz.transform(X)
        viz.finalize()
        ax.set_title(f"Rank2D — {algo}", fontsize=11, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/11_rank2d.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 11_rank2d.png")

    # ── 4.3  RadViz ───────────────────────────────────────────────────────
    print("\n[4.3] RadViz — separabilidade linear em 2D")
    fig, ax = plt.subplots(figsize=(7, 7))

    viz = RadViz(classes=class_names, features=feature_names, ax=ax)
    viz.fit(X, y)
    viz.transform(X)
    viz.finalize()
    ax.set_title("RadViz — 3 classes", fontsize=12, fontweight="bold")

    fig.savefig(f"{OUTPUT_DIR}/12_radviz.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 12_radviz.png")

    # ── 4.4  ParallelCoordinates ──────────────────────────────────────────
    print("\n[4.4] ParallelCoordinates — dados multidimensionais")
    fig, ax = plt.subplots(figsize=(12, 5))

    viz = ParallelCoordinates(
        classes=class_names,
        features=feature_names,
        sample=0.2,        # usa 20 % dos pontos para não poluir
        shuffle=True,
        ax=ax
    )
    viz.fit(X, y)
    viz.transform(X)
    viz.finalize()
    ax.set_title("ParallelCoordinates — 10 features / 3 classes", fontsize=12, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/13_parallel_coordinates.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 13_parallel_coordinates.png")

    # ── 4.5  FeatureImportances ───────────────────────────────────────────
    print("\n[4.5] FeatureImportances — impacto de cada variável")
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("FeatureImportances", fontsize=13, fontweight="bold")

    # Classificação
    clf_fi = clf(RandomForestClassifier(n_estimators=100, random_state=42))
    clf_fi.fit(X_train, y_train)
    viz_clf = FeatureImportances(clf_fi, labels=feature_names, ax=axes[0])
    viz_clf.fit(X_train, y_train)
    viz_clf.finalize()
    axes[0].set_title("Classificação (RF)", fontsize=11, fontweight="bold")

    # Regressão
    X_r, y_r = make_regression(n_samples=400, n_features=10, n_informative=6,
                                noise=20, random_state=42)
    X_tr, X_te, y_tr, y_te = train_test_split(X_r, y_r, test_size=0.25, random_state=42)
    reg_fi = reg(RandomForestRegressor(n_estimators=100, random_state=42))
    reg_fi.fit(X_tr, y_tr)
    viz_reg = FeatureImportances(reg_fi, labels=feature_names, ax=axes[1])
    viz_reg.fit(X_tr, y_tr)
    viz_reg.finalize()
    axes[1].set_title("Regressão (RF)", fontsize=11, fontweight="bold")

    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/14_feature_importances.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("   → Salvo: 14_feature_importances.png")